# Compare ablation methods

**Role:** Analysis.

Compare zeroing, learned masking and context-matched replacement using saved case-level outputs. No checkpoints are evaluated here. Intersect supported physical source cases (including graph/fate signatures) and recompute total and size-dependent summaries on common support; do not compare averages from unrelated samples. Use outputs from `size_dependent_ablation.ipynb` to obtain both views, or total-only outputs for heatmaps alone. Identical masking-trained checkpoints across all methods isolate the intervention definition; different model runs are explicitly identified as model-plus-method comparisons. All plots retain equal organoid weighting and report their common sample support.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/models/gnn.py").is_file())  # Locate the repository from the notebook’s working directory.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
ROOT = PROJECT_ROOT  # Repository root used to resolve data and result paths.

## Analysis settings

Paste the completed analysis folders here. Labels are configurable; fixed and N-dependent replacement can be separate entries. All comparisons must use the same depth and supplied count grid.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Completed outputs from total_analysis or size_dependent_ablation
ANALYSIS_DIRS = {
    'Marker zeroing': None,  # Set a completed result directory printed by either analysis notebook.
    'Masking': None,  # Prefer the same positive-rate masking checkpoint for all three methods.
    'Replacement': None,  # Add separate labels for fixed and N-dependent replacements if desired.
}
SIZE_BINS = [0,150,250,350,500,800,np.inf]  # Shared bins for observed-size comparisons.
PLOT_METRIC = 'delta_mu'  # delta_mu, delta_z, or delta_relative if saved by every run.
BOOTSTRAP_SAMPLES = 500  # Resample organoids after averaging cases/seeds within each organoid.
HEATMAP_DISPLAY_MIN_CASES = 20  # Required common unique cases, int or per-hop dict; matches inference notebooks.
MIN_PLOT_ORGANOIDS = 1  # Required organoid support per displayed pair.
SUMMARY_SEED = 42  # Bootstrap seed.
OUTPUT_DIR = ROOT/'results_experiments'/'ablation_comparison'/'default'  # Output tables and figures; choose a descriptive subfolder.


## Validate inputs and align source cases

Restrict to unchanged recipient/source identities and the same graph/fate inputs. For size sweeps require support at every N. This prevents a changing source cohort from creating artificial method differences.

Sampling is inherited from the saved analyses rather than rerun here. The table below displays the coverage budget, center/pair minimum targets, sampling seed and non-overlap settings for each method. Different sampling designs are rejected: use identical settings in total/size analysis. The heatmap threshold is applied to common retained cases after matching methods, which can be fewer than the original coverage sample.


In [ ]:
from src.analysis.interventions.fate_edits import load_ablation_result, align_method_cases, summarize_fate_effects
from src.plotting.influence_maps import plot_pair_heatmaps, plot_pair_size_curves
if len(ANALYSIS_DIRS)<2 or any(path is None for path in ANALYSIS_DIRS.values()):
    raise ValueError('Set each ANALYSIS_DIRS entry to a completed ablation output directory (or remove unused entries).')
tables,configs = {},{}
for label,path in ANALYSIS_DIRS.items():
    tables[label],configs[label] = load_ablation_result(Path(path))
    if PLOT_METRIC not in tables[label]:
        raise ValueError(f'{label} does not contain {PLOT_METRIC}.')
if len({cfg['depth'] for cfg in configs.values()})!=1:
    raise ValueError('Compare results at the same model depth.')
counts = {tuple(cfg['counts']) for cfg in configs.values()}
if len(counts)>1:
    raise ValueError('Use the same sweep counts in every result (or all total-only results).')
sampling_keys = ['sampling_scheme','centers','sampling_seed','center_sample_size',
    'center_min_marker_count','center_min_pair_count','center_apply_non_overlap','center_non_overlap_group_columns']
if any('center_sample_size' not in cfg for cfg in configs.values()):
    raise ValueError('These results predate restored coverage sampling. Rerun ablation inference with matching settings; no retraining is needed.')
sampling_settings = pd.DataFrame({label:{key:cfg.get(key) for key in sampling_keys} for label,cfg in configs.items()}).T
display(sampling_settings)
if len({json.dumps({key:cfg.get(key) for key in sampling_keys},sort_keys=True) for cfg in configs.values()}) != 1:
    raise ValueError('Use matching sampling settings across methods; inspect sampling_settings above.')
PLOT_IDENTITIES = list(dict.fromkeys(name for path,cfg in zip(ANALYSIS_DIRS.values(),configs.values())
    for name in pd.read_csv(Path(path)/'support'/f"{cfg['model_keys'][0]}_pair_coverage.csv.gz").center_marker.unique()))
checkpoint_sets = {(cfg['training_run'],tuple(cfg['model_keys'])) for cfg in configs.values()}
print('Same trained checkpoints: method comparison.' if len(checkpoint_sets)==1 else
      'Different checkpoints: effects combine ablation-method and model-training differences.')
aligned = align_method_cases(tables)
support = pd.DataFrame([dict(label=label, sampled_cases=len(table), common_rows=len(aligned[label]),
    common_organoids=aligned[label].organoid_str.nunique()) for label,table in tables.items()])
display(support)
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
(OUTPUT_DIR/'sources.json').write_text(json.dumps(configs,indent=2))
support.to_csv(OUTPUT_DIR/'common_support.csv',index=False)

sampling_settings.to_csv(OUTPUT_DIR/'sampling_settings.csv',index_label='label')


## Sample-count heatmaps on common support

Show the retained counts for each compared method after intersecting physical source cases. Each method gets an observed-size figure and, for size-dependent inputs, a sweep-cohort figure. Total observed counts pool all observed sizes; sweep counts include only the fixed cohort supported at every supplied N, counted once across N. Matching counts across methods are expected because the effects use common cases.
Rows are center identities and columns are perturbation/source identities. Each hop has its own panel. Blue intensity uses a shared logarithmic count scale, and every cell contains its integer sample count (including zero). Counts are after non-overlap and replacement-support filtering; low-support pairs remain visible here even when their effect cells are hatched. Repeated model seeds do not create additional samples.


In [ ]:
from src.analysis.interventions.fate_edits import count_fate_cases
from src.plotting.influence_maps import plot_pair_count_heatmaps
comparison_counts = pd.concat([count_fate_cases(frame,metric=PLOT_METRIC).assign(label=label)
    for label,frame in aligned.items()],ignore_index=True)
comparison_counts.to_csv(OUTPUT_DIR/'pair_sample_counts.csv',index=False)
count_limit = max(2,float(comparison_counts.n_cases.max())) if not comparison_counts.empty else 2
count_hops = range(1,next(iter(configs.values()))['depth']+1)
count_modes = ['observed','sweep'] if next(iter(configs.values()))['counts'] else ['observed']
for index,label in enumerate(aligned):
    for mode in count_modes:
        selected = comparison_counts[(comparison_counts.label==label)&(comparison_counts.analysis==mode)]
        description = 'pooled observed sizes' if mode=='observed' else 'fixed all-N sweep cohort'
        fig = plot_pair_count_heatmaps(selected,hops=count_hops,marker_names=PLOT_IDENTITIES,vmax=count_limit,
            title=f'{label}: common retained samples, {description}')
        fig.savefig(OUTPUT_DIR/f'pair_sample_counts_{index}_{mode}.png',dpi=160,bbox_inches='tight')
        plt.show()


## Total comparison

Pool observed-size effects across organoids on the shared case set. Heatmaps share a symmetric color scale across methods; no hop-zero intervention is included.

In [ ]:
(OUTPUT_DIR/'plot_settings.json').write_text(json.dumps(dict(
    metric=PLOT_METRIC,heatmap_display_min_cases=HEATMAP_DISPLAY_MIN_CASES,
    min_plot_organoids=MIN_PLOT_ORGANOIDS,bootstrap_samples=BOOTSTRAP_SAMPLES),indent=2))
total = pd.concat([summarize_fate_effects(frame,view='total',metric=PLOT_METRIC,
    draws=BOOTSTRAP_SAMPLES,seed=SUMMARY_SEED).assign(label=label) for label,frame in aligned.items()],ignore_index=True)
total.to_csv(OUTPUT_DIR/'total_comparison.csv',index=False)
limit = float(total['mean'].abs().max()) or 1.
for label,part in total.groupby('label',sort=False):
    fig = plot_pair_heatmaps(part,title=f'{label}: {PLOT_METRIC}, common source cases',
        min_organoids=MIN_PLOT_ORGANOIDS,min_cases=HEATMAP_DISPLAY_MIN_CASES,marker_names=PLOT_IDENTITIES,limit=limit,hops=range(1,next(iter(configs.values()))['depth']+1))
    fig.savefig(OUTPUT_DIR/f'total_{list(aligned).index(label)}.png',dpi=160,bbox_inches='tight')
    plt.show()
display(total)


## Size-dependent comparison

Overlay methods within each recipient/source pair and exact hop. Solid curves group different observed organoids by N; dashed curves sweep N on fixed graphs. Bands are pointwise organoid-bootstrap intervals conditional on the fitted models.

Observed and sweep curves use distinct colors, with matching uncertainty bands. Observed curves also use solid lines and circles; sweeps use dashed lines and squares. Each figure title identifies the exact hop distance.


In [ ]:
if next(iter(configs.values()))['counts']:
    size = pd.concat([summarize_fate_effects(frame,view='size',metric=PLOT_METRIC,size_bins=SIZE_BINS,
        draws=BOOTSTRAP_SAMPLES,seed=SUMMARY_SEED).assign(label=label) for label,frame in aligned.items()],ignore_index=True)
    size.to_csv(OUTPUT_DIR/'size_comparison.csv',index=False)
    for hop in sorted(size.hop.unique()):
        fig = plot_pair_size_curves(size,hop=hop,title=f'Common source cases: {PLOT_METRIC}',
            min_organoids=MIN_PLOT_ORGANOIDS,min_cases=HEATMAP_DISPLAY_MIN_CASES)
        fig.savefig(OUTPUT_DIR/f'size_hop{hop}.png',dpi=160,bbox_inches='tight')
        plt.show()
else:
    print('Selected total-only results. Select outputs of size_dependent_ablation for size comparisons.')
